# Exploración de modelos multimodales: construyendo un mini-CLIP desde cero

**Objetivo:** entender cómo funcionan y de qué se componen los modelos multimodales
(imagen + texto) construyendo una versión chiqueta de CLIP, pieza por pieza, sin cajas negras.

**Cómo usar este notebook:** ejecutar las celdas **en orden**. Cada sección construye la pieza
que la siguiente necesita. Al final tenemos un modelo entrenado que hace *zero-shot
classification*, y comparamos con CLIP real.

| Sección | Qué construimos | Concepto clave |
|---|---|---|
| 1 | Mapa conceptual | Dos torres + espacio compartido |
| 2 | Codificador de texto | Tokens → embeddings → atención |
| 3 | Codificador de visión | Imagen = "frase de parches" (ViT) |
| 4 | El puente | Proyección al espacio compartido |
| 5 | El objetivo de entrenamiento | Pérdida contrastiva (InfoNCE) |
| 6 | Dataset + entrenamiento | Datos sintéticos, loop de entrenamiento |
| 7 | Demo | Clasificación zero-shot |
| 8 | Conexión a la realidad | CLIP preentrenado y VLMs completos |

> **Para tu compañera:** este notebook es la puerta de entrada al proyecto. Si entendés
> las secciones 1-5, entendés el 80% de cómo los modelos multimodales modernos
> (CLIP, LLaVA, GPT-4V) estructuran la relación imagen-texto.


## 1. Mapa conceptual: qué significa "multimodal"

Un modelo multimodal procesa **más de un tipo de dato** (modalidades): imágenes, texto,
audio, video. Nosotros nos concentramos en el par más estudiado: **imagen + texto**.

La arquitectura dominante tiene una estructura en tres partes:

```
        IMAGEN                        TEXTO
          |                             |
    [ Vision Encoder ]           [ Text Encoder ]
    (ViT: parches->tokens)       (tokens -> embeddings)
          |                             |
          +--------> [ ESPACIO COMPARTIDO ] <----+
                    (un solo espacio de vectores
                     donde imagen y texto se comparan)
```

**La idea central:** cada modalidad tiene su propio codificador (su "torre"), pero ambos
terminan produciendo **vectores del mismo tipo y tamaño** en un espacio compartido.
Ahí adentro, una imagen y su descripción quedan *cerca* (alta similitud coseno), y una
imagen y un texto que no la describen quedan *lejos*.

Existen dos grandes familias:

- **Contrastivas (CLIP, SigLIP):** entrenan el espacio compartido para acercar pares
  correctos imagen-texto y alejar los incorrectos. No generan texto; *entienden* la relación.
- **Generativas (LLaVA, GPT-4V, Qwen-VL):** toman un encoder visual (¡frecuentemente un
  CLIP preentrenado!), lo *proyectan* al espacio del LLM, y el LLM genera texto condicionado
  en la imagen.

> **Para tu compañera:** nuestro proyecto (diagnóstico de *greedy modality*) vive justo acá:
> cuando un modelo multimodal aprende a apoyarse casi solo en una modalidad (normalmente
> texto) e ignora la otra, el pipeline de dos torres está fallando en su promesa de
> *fusionar*. Para diagnosticarlo primero hay que construir bien las torres, y eso es
> exactamente lo que hacemos en este notebook.


## 2. Lado texto: de palabras a vectores

Una red neuronal no entiende strings; entiende números. El pipeline de texto tiene 3 pasos:

1. **Tokenización:** cortar el texto en piezas (tokens) y mapearlas a enteros (IDs).
2. **Embedding:** cada ID se convierte en un vector denso que se *aprende* durante el entrenamiento.
3. **Codificación con contexto:** un Transformer procesa la secuencia de embeddings para que
   cada token "vea" a los demás (atención).

Acá construimos un tokenizador **a mano** (vocabulario chico, por palabras). Los modelos
reales usan subword tokenizers (BPE, WordPiece), pero el principio es el mismo: texto → IDs.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"torch {torch.__version__} | numpy {np.__version__}")

### 2.1 Tokenizador a mano

Nuestro dataset va a ser de figuras con colores, y las descripciones usan un vocabulario
cerrado de 8 palabras. Con eso basta un diccionario palabra → ID. El token `<pad>` (ID 0)
sirve para rellenar secuencias cortas hasta un largo fijo, y después le diremos a la
atención que lo *ignore*.


In [ ]:
CAPTION_WORDS = ["un", "círculo", "cuadrado", "triángulo", "rojo", "azul", "verde"]

def build_vocab(words):
    """Assign integer IDs: 0 reserved for <pad>, words get 1..N."""
    return {"<pad>": 0, **{w: i + 1 for i, w in enumerate(sorted(words))}}

def tokenize(captions, vocab):
    """Convert captions to a padded (N, max_len) array of token IDs."""
    max_len = max(len(c.split()) for c in captions)
    ids = np.zeros((len(captions), max_len), dtype=np.int64)
    for i, cap in enumerate(captions):
        for j, word in enumerate(cap.lower().split()):
            ids[i, j] = vocab[word]  # KeyError here = word outside vocab
    return ids

vocab = build_vocab(CAPTION_WORDS)
print("Vocabulario:", vocab)

demo = tokenize(["un círculo rojo", "un triángulo verde"], vocab)
print("IDs de 'un círculo rojo':     ", demo[0])
print("IDs de 'un triángulo verde': ", demo[1])

### 2.2 El bloque Transformer, desde cero

Esta es la pieza más importante del notebook, y la construimos **sin librerías de
transformers**: atención multi-cabeza + MLP, con normalización y residuales.

**Atención en una frase:** cada token genera tres vectores — *Query* ("qué busco"),
*Key* ("qué ofrezco") y *Value* ("qué entrego") — y cada token se reconstruye como un
promedio ponderado de los *Values* de todos los tokens, donde los pesos salen de comparar
su *Query* contra los *Keys* (producto punto escalado + softmax).

Detalles que van a aparecer en el código:

- **Multi-cabeza:** partimos el vector en `n_heads` trozos y hacemos atención en paralelo
  en cada trozo. Cada cabeza puede especializarse en relaciones distintas.
- **Escala por sqrt(d_head):** sin dividir por `sqrt(d_head)`, los productos punto crecen
  con la dimensión y el softmax satura (gradientes casi nulos).
- **Pre-norm + residual:** `x = x + Attn(LN(x))`. El residual es el "camino rápido" que
  deja que la información fluya aunque una capa no aporte nada. Es lo que hace entrenables
  a las redes profundas.
- **Dropout:** los transformers reales usan ~0.1; en nuestro juguete lo dejamos en 0
  porque con tan pocos datos y pasos, solo frenaría la convergencia sin aportar regularización.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)  # one matrix for Q, K, V
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, pad_mask=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Reshape to (B, n_heads, T, d_head): attention runs per head
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product: how much each token attends to each token
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, h, T, T)
        if pad_mask is not None:  # True where padding -> hide those keys
            scores = scores.masked_fill(pad_mask[:, None, None, :], float("-inf"))
        attn = scores.softmax(dim=-1)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(out)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block: attention + MLP, both with residuals."""

    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.0):
        # dropout=0.0: for a tiny toy model it only slows convergence.
        # Real transformers use ~0.1.
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, pad_mask=None):
        x = x + self.attn(self.ln1(x), pad_mask=pad_mask)
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
class TextEncoder(nn.Module):
    """Token IDs -> embeddings -> Transformer blocks -> one vector per caption."""

    def __init__(self, vocab_size, d_model=64, n_layers=2, n_heads=4, max_len=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        pad_mask = token_ids == 0  # <pad> has ID 0
        x = self.embed(token_ids) + self.pos[:, : token_ids.shape[1]]
        for block in self.blocks:
            x = block(x, pad_mask=pad_mask)
        x = self.ln(x)
        # Mean-pool over real (non-pad) tokens -> single vector per caption
        keep = (~pad_mask).float()[..., None]
        return (x * keep).sum(dim=1) / keep.sum(dim=1).clamp(min=1.0)


# Smoke test: check input/output shapes flow through the whole encoder
text_encoder = TextEncoder(vocab_size=len(vocab), d_model=64, n_layers=2, n_heads=4)
with torch.no_grad():
    out = text_encoder(torch.from_numpy(demo))
print("Entrada:", demo.shape, "-> salida del codificador de texto:", tuple(out.shape))

## 3. Lado visión: la imagen como "frase de parches"

**Pregunta frecuente: ¿acá van redes convolucionales (CNN) con filtros?**

Respuesta corta: **puede ser cualquiera de las dos, pero el estándar moderno es el ViT**
(Vision Transformer) y es el que construimos acá. El contexto:

- **Histórico:** hasta 2020, la visión era territorio de las CNN. Filtros convolucionales
  chicos barrían la imagen detectando patrones locales (bordes → texturas → partes →
  objetos), con una jerarquía construida capa a capa. El CLIP original de OpenAI salió con
  ** ambas variantes**: torres ResNet (CNN) y torres ViT.
- **Actual:** desde *An Image is Worth 16x16 Words* (Dosovitskiy et al., 2020), el ViT
  ganó. ¿Por qué? Porque **convierte la imagen en una secuencia de tokens**, igual que el
  texto. Y cuando ambas modalidades son secuencias de tokens, **la misma máquina
  (nuestro `TransformerBlock` de la sección 2) procesa a las dos**. Esa simetría es lo que
  hace natural el puente multimodal.

Cómo funciona el *patchify*:

```
 Imagen 32x32  ->  cortar en parches 8x8  ->  16 parches  ->  cada parche = 1 token
 [ | | | ]        como si fuera una            [t1 t2 t3 t4]
 [ | | | ]        grilla de "palabras"          + positional embedding
 [ | | | ]        visuales                      (sin orden, el transformer
 [ | | | ]                                         no sabe dónde estaba cada parche)
```

**Dato sabroso:** la proyección lineal de cada parche a un vector se implementa (y se
piensa) como **una convolución con kernel = stride = tamaño de parche**. O sea: los
"filtros" no desaparecieron del todo, solo se redujeron a un solo paso de entrada; la
modelación de relaciones entre regiones se la delega a la atención.

> **Para tu compañera:** CNN y ViT resuelven lo mismo con inductivas distintas: la CNN
> *asume* localidad y translation-equivariance (menos datos necesarios); el ViT no asume
> casi nada y aprende las relaciones desde los datos (necesita más datos, pero escala
> mejor). En multimodal gana ViT por la simetría con texto.


In [ ]:
class PatchEmbed(nn.Module):
    """Split the image into patches and project each patch to a d_model vector.
    Implemented as a Conv2d with kernel_size = stride = patch_size."""

    def __init__(self, img_size=32, patch_size=8, in_channels=3, d_model=64):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, img):
        x = self.proj(img)                    # (B, d_model, H/p, W/p)
        return x.flatten(2).transpose(1, 2)   # (B, n_patches, d_model)


class VisionEncoder(nn.Module):
    """Image -> patches -> Transformer blocks -> one vector per image."""

    def __init__(self, img_size=32, patch_size=8, d_model=64, n_layers=2, n_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, d_model=d_model)
        n_patches = self.patch_embed.n_patches
        self.pos = nn.Parameter(torch.zeros(1, n_patches, d_model))
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, img):
        x = self.patch_embed(img) + self.pos  # add positional info to patches
        for block in self.blocks:
            x = block(x)
        x = self.ln(x)
        return x.mean(dim=1)  # mean-pool over patches (CLIP uses a [CLS] token)


# Smoke test: one image through the whole vision encoder
vision_encoder = VisionEncoder(d_model=64, n_layers=2, n_heads=4)
with torch.no_grad():
    out = vision_encoder(torch.rand(2, 3, 32, 32))
print("Entrada: (2, 3, 32, 32) -> salida del codificador de visión:", tuple(out.shape))

## 4. El puente: proyección al espacio compartido

Las dos torres terminan en vectores de 64 dims, pero viven en **espacios distintos**: no
podemos comparar el vector de un texto con el de una imagen así porque nada obliga a que
estén en la misma "regla". El puente son **dos proyecciones lineales** (una por torre) que
lleven todo a un espacio compartido de dimensión `embed_dim`, y una **normalización L2**
que deje todos los vectores sobre la esfera unitaria.

¿Por qué normalizar? Porque con vectores sobre la esfera, el **producto punto ES la
similitud coseno**: un número en [-1, 1] que mide ángulo, no magnitud. Eso hace que la
comparación sea estable y comparable entre ejemplos.

```
 imagen --[VisionEncoder]--> v_img (64) --[W_img]--> z_img (64, normalizado)
                                                              \
                                                               producto punto
                                                              /          \
 texto --[TextEncoder]-----> v_txt (64) --[W_txt]--> z_txt (64, normalizado)
```


In [ ]:
class MiniCLIP(nn.Module):
    """Two towers + two projection heads -> one shared embedding space."""

    def __init__(self, vocab_size, d_model=64, embed_dim=64,
                 n_layers=2, n_heads=4):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, d_model, n_layers, n_heads)
        self.vision_encoder = VisionEncoder(d_model=d_model, n_layers=n_layers,
                                            n_heads=n_heads)
        self.text_proj = nn.Linear(d_model, embed_dim)
        self.image_proj = nn.Linear(d_model, embed_dim)
        # Temperature: learned, starts at CLIP's 1/0.07. Multiplies similarities
        # before softmax; low temp = sharper decisions.
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, img):
        return F.normalize(self.image_proj(self.vision_encoder(img)), dim=-1)

    def encode_text(self, token_ids):
        return F.normalize(self.text_proj(self.text_encoder(token_ids)), dim=-1)

    def forward(self, img, token_ids):
        return self.encode_image(img), self.encode_text(token_ids)


model = MiniCLIP(vocab_size=len(vocab), d_model=64, embed_dim=64).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"MiniCLIP inicializado: {n_params:,} parámetros (CLIP real: ~150 millones)")

## 5. El objetivo de entrenamiento: pérdida contrastiva (InfoNCE)

¿Cómo entrenamos el espacio compartido? Con un truco elegante. Tomamos un batch de B pares
(imagen, descripción) **que se corresponden** y calculamos **todas** las similitudes
cruzadas: una matriz B×B.

```
                texto_1   texto_2   texto_3
    imagen_1 [   0.92      0.11      0.05 ]   <- la diagonal son los pares REALES
    imagen_2 [   0.08      0.87      0.14 ]
    imagen_3 [   0.03      0.19      0.90 ]
```

La matriz de similitudes **ya es una matriz de logits de clasificación**: para la imagen_1,
la "clase correcta" es texto_1, para imagen_2 es texto_2... O sea, entrenar el
alineamiento es literalmente hacer **cross-entropy sobre la diagonal**. Eso es InfoNCE:

- **i2t:** dada una imagen, elegir su texto correcto entre los B del batch → `cross_entropy(logits, arange(B))`.
- **t2i:** simétrico, dado un texto elegir su imagen → `cross_entropy(logits.T, arange(B))`.
- Los B-1 pares incorrectos de la matriz actúan como **ejemplos negativos** gratis: sin
  dataset de "pares malos", los negativos salen del propio batch.

**La temperatura** (`logit_scale`) escala los logits antes del softmax: controla qué tan
punitive es la pérdida con los negativos. CLIP la aprende durante el entrenamiento.

> **Sanity check incluido abajo:** con embeddings aleatorios sin entrenar y **temperatura 1**,
> todos los logits son casi 0, el softmax es casi uniforme y la pérdida debe dar ≈ `ln(B)`
> (adivinar 1 entre B opciones al azar). Si no da eso, la pérdida está mal implementada.


In [ ]:
def clip_loss(image_emb, text_emb, logit_scale):
    """Symmetric InfoNCE: cross-entropy over the similarity matrix diagonal."""
    logits = logit_scale.exp().clamp(max=100.0) * image_emb @ text_emb.t()
    labels = torch.arange(len(logits), device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)      # image -> its text
    loss_t2i = F.cross_entropy(logits.t(), labels)  # text -> its image
    return (loss_i2t + loss_t2i) / 2, logits


# Sanity check: with temperature scale = 1 (log(1) = 0) and random embeddings,
# every logit is ~= 0 -> uniform softmax -> loss should be ~= ln(B)
B = 16
logit_scale_check = torch.tensor(0.0)
rand_img = F.normalize(torch.randn(B, 64), dim=-1)
rand_txt = F.normalize(torch.randn(B, 64), dim=-1)
loss_rand, _ = clip_loss(rand_img, rand_txt, logit_scale_check)
print(f"Pérdida con embeddings aleatorios y temperatura 1: {loss_rand.item():.3f}")
print(f"Valor teórico ln({B}) = {math.log(B):.3f}  <- deben coincidir")

## 6. Dataset sintético + entrenamiento

Para *aprender* cómo se construye esto, el dataset ideal es uno que: (a) corra en minutos
en CPU, (b) tenga un alineamiento imagen-texto **obvio y verificable**. Generamos figuras
geométricas coloreadas (3 formas × 3 colores) con posiciones y tamaños aleatorios, cada una
con su descripción: *"un círculo rojo"*.

Si el modelo aprende, va a poder decir qué figura ve **sin haberle enseñado etiquetas**,
solo por la cercanía en el espacio compartido. Eso es exactamente el modo *zero-shot* de
CLIP, en versión de juguete.

**Nota de diseño (lección empírica):** generamos 320 ejemplos por combinación (2880 en
total) en vez de unos pocos. ¿Por qué? Cada imagen tiene posición y tamaño aleatorios, así
que dos instancias de "un círculo rojo" nunca son idénticas: esa varianza intra-clase es un
piso de ruido que **ninguna arquitectura elimina**. Lo que sí se puede hacer es cubrir mejor
el espacio de variaciones con más datos. Lo verificamos midiendo: con 720 ejemplos el
accuracy zero-shot era ~82%; con 2880 (y el mismo modelo chico) llega a ~99%. **Primero la
escala de datos, después la capacidad del modelo** — exactamente la tesis de CLIP.


In [ ]:
SHAPES = ["círculo", "cuadrado", "triángulo"]
COLORS = {"rojo": [0.85, 0.15, 0.12], "azul": [0.15, 0.30, 0.85], "verde": [0.12, 0.65, 0.25]}


def draw_shape(shape, color_name, img_size=32):
    """Render one colored shape on a light background, with noise and jitter."""
    img = np.full((img_size, img_size, 3), 0.95, dtype=np.float32)
    color = np.clip(np.array(COLORS[color_name]) * rng.uniform(0.85, 1.1), 0, 1)
    cx, cy = rng.integers(10, img_size - 10, size=2)  # center jitter
    r = int(rng.integers(5, 9))                        # size jitter
    yy, xx = np.mgrid[0:img_size, 0:img_size]

    if shape == "círculo":
        mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= r**2
    elif shape == "cuadrado":
        mask = (np.abs(yy - cy) <= r) & (np.abs(xx - cx) <= r)
    else:  # triangle: inside = same sign for all 3 edge cross-products
        a, b, c = (cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)
        def edge(p, q):
            return (q[0] - p[0]) * (yy - p[1]) - (q[1] - p[1]) * (xx - p[0])
        e1, e2, e3 = edge(a, b), edge(b, c), edge(c, a)
        mask = ((e1 >= 0) & (e2 >= 0) & (e3 >= 0)) | ((e1 <= 0) & (e2 <= 0) & (e3 <= 0))

    img[mask] = color
    img += rng.normal(0, 0.03, img.shape).astype(np.float32)  # pixel noise
    return np.clip(img, 0.0, 1.0)


def make_split(n_per_combo):
    """n_per_combo examples for each (shape, color) pair."""
    imgs, captions = [], []
    for shape in SHAPES:
        for color in COLORS:
            for _ in range(n_per_combo):
                imgs.append(draw_shape(shape, color))
                captions.append(f"un {shape} {color}")
    return np.stack(imgs), captions


train_imgs, train_caps = make_split(n_per_combo=320)
test_imgs, test_caps = make_split(n_per_combo=10)
print(f"Train: {len(train_imgs)} pares | Test: {len(test_imgs)} pares")

# Visual check: always look at your data before training
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, cap in zip(axes.flat, train_imgs, train_caps):
    ax.imshow(img)
    ax.set_title(cap, fontsize=10)
    ax.axis("off")
plt.suptitle("Muestras del dataset sintético")
plt.tight_layout()
plt.show()

In [ ]:
# --- Preprocess: images to tensors, captions to padded token IDs ---
train_imgs_t = torch.from_numpy(train_imgs).permute(0, 3, 1, 2).float().to(device)
train_ids_t = torch.from_numpy(tokenize(train_caps, vocab)).to(device)

EPOCHS, BATCH = 30, 64
# Contrastive training needs many steps to sharpen the shared space: CLIP
# trained for hundreds of thousands of steps with a cosine learning-rate
# schedule. Here: 2880 pairs / batch 64 = 45 steps x 30 epochs, ~1-2 min CPU.
# (More data beats more epochs or a bigger model in this regime.)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
n = len(train_imgs_t)
loss_history = []

model.train()
for epoch in range(1, EPOCHS + 1):
    perm = torch.randperm(n, device=device)
    epoch_loss = 0.0
    for start in range(0, n, BATCH):
        idx = perm[start : start + BATCH]
        img_emb, txt_emb = model(train_imgs_t[idx], train_ids_t[idx])
        loss, _ = clip_loss(img_emb, txt_emb, model.logit_scale)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    loss_history.append(epoch_loss / n)
    scheduler.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}/{EPOCHS} | loss = {loss_history[-1]:.4f} "
              f"| lr = {scheduler.get_last_lr()[0]:.2e}")

model.eval()
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("epoch")
plt.ylabel("InfoNCE loss")
plt.title("Curva de entrenamiento (referencia: ln(64) = 4.16 al inicio)")
plt.grid(alpha=0.3)
plt.show()

## 7. Demo: clasificación zero-shot

Acá está la magia de CLIP hecha visible. **Nunca entrenamos un clasificador**: no hay capa
final con 6 clases ni etiquetas numéricas. Clasificar es:

1. Codificar los 9 prompts posibles ("un círculo rojo", ..., "un triángulo verde") con la torre de texto.
2. Codificar la imagen de test con la torre de visión.
3. Elegir el prompt con **mayor similitud coseno**.

El "clasificador" se escribe en lenguaje natural en tiempo de inferencia. Si cambiás las
clases, no reentrenás nada: cambiás los prompts. Esa es LA propiedad que hizo a CLIP tan
influyente.

Evaluamos sobre el split de test (figuras que el modelo nunca vio) y miramos la matriz de
confusión para ver *dónde* se confunde.


In [ ]:
# --- Build the "classifier" out of natural language prompts ---
prompts = [f"un {s} {c}" for s in SHAPES for c in COLORS]
prompt_names = [f"{s} {c}" for s in SHAPES for c in COLORS]
with torch.no_grad():
    prompt_ids = torch.from_numpy(tokenize(prompts, vocab)).to(device)
    prompt_emb = model.encode_text(prompt_ids)          # (6, embed_dim)

    test_imgs_t = torch.from_numpy(test_imgs).permute(0, 3, 1, 2).float().to(device)
    test_emb = model.encode_image(test_imgs_t)          # (60, embed_dim)
    sim = test_emb @ prompt_emb.t()                     # (N, n_cls) cosine similarities
    pred = sim.argmax(dim=1).cpu().numpy()

# Ground truth: captions were generated in the same (shape, color) order
n_cls = len(prompts)
true = np.repeat(np.arange(n_cls), len(test_imgs) // n_cls)
acc = (pred == true).mean()
print(f"Zero-shot accuracy en test: {acc:.1%}")

# Confusion matrix: rows = true class, cols = predicted
cm = np.zeros((n_cls, n_cls), dtype=int)
for t, p in zip(true, pred):
    cm[t, p] += 1
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_cls), prompt_names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n_cls), prompt_names, fontsize=8)
ax.set_xlabel("predicho")
ax.set_ylabel("real")
ax.set_title("Matriz de confusión zero-shot")
for i in range(n_cls):
    for j in range(n_cls):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)
plt.tight_layout()
plt.show()

# Qualitative: show a few predictions
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
sample = rng.integers(0, len(test_imgs), size=8)
for ax, k in zip(axes.flat, sample):
    ax.imshow(test_imgs[k])
    ok = pred[k] == true[k]
    ax.set_title(f"real: {prompt_names[true[k]]}\npred: {prompt_names[pred[k]]}",
                 fontsize=9, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("Predicciones zero-shot (verde = correcto)")
plt.tight_layout()
plt.show()

## 8. Conexión a la realidad: CLIP real y los VLMs completos

Nuestro mini-CLIP tiene ~200 mil parámetros; CLIP real tiene ~150 millones, fue entrenado
con **400 millones de pares** imagen-texto de internet, y su torre de visión es un ViT
exactamente como el nuestro, pero con 12 capas y parches de 32x32 píxeles (¡por eso se
llama ViT-B/32!). **La arquitectura conceptual es la misma que acabamos de construir.**

La celda opcional de abajo carga CLIP preentrenado (requiere `transformers` instalado y
~600 MB de descarga la primera vez). Un detalle pedagógico: usamos prompts **en inglés**
porque CLIP fue entrenado mayormente con texto en inglés — el alineamiento imagen-texto
aprende la *distribución* del texto con el que fue entrenado.

### ¿Y los modelos que además generan texto (LLaVA, GPT-4V)?

```
                [ Vision Encoder ]      (ej: ViT de CLIP, congelado o fino)
                       |
                 [ Proyector ]          (capa(s) que llevan vectores visuales
                       |                 al espacio de embeddings del LLM)
                       v
   prompt: "¿Qué hay en esta imagen?"  +  tokens visuales proyectados
                       |
                 [ LLM decoder ]   -->  "Un círculo rojo sobre fondo claro"
```

Un VLM generativo es: **encoder visual + proyector + LLM**, entrenado con next-token
prediction sobre conversaciones con imágenes. Frecuentemente el encoder visual es un CLIP
preentrenado: **todo lo que construimos acá es la primera mitad de un VLM completo.**

> **Para tu compañera (y cierre de ciclo con el proyecto):** el fenómeno de *greedy
> modality* ocurre cuando el modelo, en algún punto de este pipeline, aprende que puede
> resolver la tarea apoyándose casi solo en una torre (típicamente el texto, que suele ser
> más "fácil") y la otra queda subutilizada. Nuestro marco estadístico busca **medir** ese
> desbalance y **corregirlo**. Este notebook es el fundamento: si entendés cómo se
> construyen las torres y el puente, entendés exactamente *dónde* puede quedar la grieta.


In [ ]:
# [OPTIONAL] Requires: pip install transformers  (+ ~600 MB download, internet)
# Skipped by the smoke test; run manually when you want to see the real thing.
try:
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor
except ImportError as e:
    print(f"Falta una dependencia ({e.name}). Instalá con: pip install transformers pillow")
else:
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()

    # English prompts: CLIP was trained mostly on English image-text pairs
    en_prompts = [f"a {s} {c}" for s in ["circle", "square", "triangle"]
                  for c in ["red", "blue", "green"]]

    # Feed a few of OUR synthetic images to the real CLIP
    sample_idx = rng.integers(0, len(test_imgs), size=4)
    pil_imgs = [Image.fromarray((test_imgs[k] * 255).astype("uint8"))
                for k in sample_idx]

    with torch.no_grad():
        inputs = clip_processor(text=en_prompts, images=pil_imgs,
                                return_tensors="pt", padding=True)
        outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)

    for img, row in zip(pil_imgs, probs):
        top = row.argmax()
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.imshow(img)
        ax.set_title(f"CLIP real: '{en_prompts[top]}' ({row[top]:.0%})", fontsize=10)
        ax.axis("off")
        plt.show()

## Cierre: lo que acabamos de construir

| Pieza | Qué hace | En CLIP real |
|---|---|---|
| Tokenizador | texto → IDs | BPE (50k tokens) |
| `TextEncoder` | IDs → vector con contexto | Transformer 12 capas |
| `PatchEmbed` | imagen → 16 parches → tokens | conv 32x32 stride 32 |
| `VisionEncoder` | parches → vector con contexto | ViT-B (12 capas, 12 cabezas) |
| Proyecciones | llevar ambas torres al espacio compartido | igual, con dims ~512-768 |
| `clip_loss` | InfoNCE simétrico sobre la diagonal | idéntico, con batches enormes (32k) |
| Zero-shot | clasificar por similitud con prompts | exactamente igual |

**Glosario mínimo:**

- **Embedding:** vector denso que representa un dato (palabra, parche, imagen, frase).
- **Espacio compartido:** espacio vectorial donde ambas modalidades viven y se comparan.
- **Similitud coseno:** producto punto de vectores normalizados; 1 = misma dirección.
- **InfoNCE / pérdida contrastiva:** cross-entropy sobre la diagonal de la matriz de similitudes.
- **Zero-shot:** resolver una tarea sin ejemplos de entrenamiento de esa tarea, solo con prompts.
- **VLM (Vision-Language Model):** modelo generativo = encoder visual + proyector + LLM.
- **Greedy modality:** cuando el modelo se apoya desproporcionadamente en una modalidad
  e ignora la otra — el fenómeno que nuestro proyecto va a medir y corregir.

**Próximos pasos naturales (para el proyecto):**

1. Experimentar: cambiar `d_model`, `n_layers`, tamaño de batch → observar el efecto en la pérdida.
2. Romper a propósito: entrenar solo con una clase de color y ver qué aprende (sesgo).
3. Diseñar métricas de dependencia por modalidad sobre un modelo así → semilla del marco estadístico.
